# 01 — Retrieval metrics

Reproduce the article's Recall@k / MRR / nDCG@k example, then run the same metrics on a small
scifact slice with the dense retriever.


In [ ]:
from rag_evals.evaluation.retrieval import evaluate_runs

gold = {
    "q1": {"d3"},
    "q2": {"d7", "d2"},
    "q3": {"d11"},
    "q4": {"d5"},
}
runs = {
    "q1": ["d8", "d3", "d1", "d4", "d2", "d9", "d6", "d10", "d12", "d13"],
    "q2": ["d2", "d6", "d4", "d7", "d1", "d3", "d8", "d11", "d5", "d9"],
    "q3": ["d11", "d2", "d3", "d4", "d1", "d6", "d7", "d8", "d10", "d12"],
    "q4": ["d1", "d2", "d3", "d6", "d8", "d9", "d10", "d12", "d13", "d14"],
}
m = evaluate_runs(runs, gold, k=5)
print(f"Recall@5 = {m.recall_at_k:.3f}  (article: 0.750)")
print(f"MRR      = {m.mrr:.3f}  (article: 0.625)")
print(f"nDCG@5   = {m.ndcg_at_k:.3f}  (article: 0.627)")


Now apply the same metrics to live retrieval against scifact.


In [ ]:
import json
from pathlib import Path
from rag_evals.config import settings

retrieval_path = settings.golden_dir / "retrieval.jsonl"
rows = [json.loads(l) for l in retrieval_path.open()][:50]  # smoke subset
print(f"loaded {len(rows)} queries from {retrieval_path.name}")


In [ ]:
from rag_evals.retrieval.dense import DenseRetriever

dense = DenseRetriever()
runs_dense, gold = {}, {}
for r in rows:
    hits = dense(r["query"], limit=30)
    runs_dense[r["qid"]] = [h.doc_id for h in hits]
    gold[r["qid"]] = r["gold_doc_ids"]

m = evaluate_runs(runs_dense, gold, k=10)
print(f"dense Recall@10 = {m.recall_at_k:.3f}")
print(f"dense MRR       = {m.mrr:.3f}")
print(f"dense nDCG@10   = {m.ndcg_at_k:.3f}")
